# 05 — Calibration, PSI và CSI

Ba khối trước đo xếp hạng: Gini, KS, đóng góp biên. Khối này đo hai thứ khác hẳn.

Một, **PD có phải con số thật không**. Gini chỉ nói model xếp đúng thứ tự, không nói 5% là 5% hay 15%. Ngân hàng dùng PD để định giá khoản vay và trích lập dự phòng, nên một model xếp hạng giỏi mà nói sai mức thì vẫn hỏng.

Hai, **model còn dùng được không khi chưa có nhãn**. Nhãn "trễ 90+ ngày trong 2 năm" đến sau 12 đến 24 tháng. Trong khoảng đó PSI là thứ duy nhất trả lời được, và nó trả lời bằng cách nhìn đầu vào, không nhìn kết quả.

Hai quyết định chốt trước khi chạy phép đo nào:

- Lớp hiệu chỉnh **fit trên `test`, đánh giá trên `oot`**. Test đã dùng ở khối 3 và 4 nhưng chỉ để báo cáo, không để chọn gì. Cách này giữ `oot` sạch cho con số cuối, và nó cũng đúng hình dạng thật của việc triển khai: calibrator fit trên dữ liệu cũ rồi áp lên kỳ mới.
- Mười bốn dự đoán ghi ở `notes/du_doan_khoi5.md` **trước khi đọc nhãn `oot` lần nào**.

Đây là lần đầu tiên tập `oot` được đọc nhãn trong cả dự án.

In [1]:
import sys
from pathlib import Path
import numpy as np, pandas as pd, sqlite3

sys.path.insert(0, str(Path.cwd().parent / 'src'))
import config, scorecard, tree_model, calibration as cal, psi as P

pd.set_option('display.width', 220); pd.set_option('display.max_columns', 60)

bins_m, woe_m, _ = scorecard.load_data(mono=True)
Wm = scorecard.woe_matrix(bins_m, woe_m)
tr, te, oo = [(bins_m.split == s).values for s in ('train', 'test', 'oot')]
lr, coef, se = scorecard.fit_logit(Wm[tr], (1 - bins_m.target)[tr])
y = bins_m.target.values                     # 1 = BAD, quy uoc rieng cua khoi nay
pd_sc = 1 - lr.predict_proba(Wm)[:, 1]       # PD cua scorecard, moi dong

con = sqlite3.connect(scorecard._ro_uri(config.DB_PATH), uri=True)
ap = pd.read_sql('SELECT * FROM applications', con).sort_values('id').reset_index(drop=True)
fw = pd.read_sql('SELECT * FROM features_woe_mono', con).sort_values('id').reset_index(drop=True)
con.close()
assert (bins_m.index.values == ap.id.values).all(), 'thu tu dong khong khop'

import xgboost as xgb
Xs, split, yv = tree_model.feature_sets(ap, fw)
Z4 = pd.concat([Xs['M3_goc'], ap[['open_credit_lines', 'debt_ratio']]], axis=1)
P4 = dict(max_depth=3, learning_rate=0.10, n_estimators=250,
          min_child_weight=50, subsample=0.8, colsample_bytree=0.8)
mx = xgb.XGBClassifier(tree_method='hist', eval_metric='auc',
                       random_state=config.SEED, **P4).fit(Z4[tr], yv[tr])
pd_xgb = 1 - mx.predict_proba(Z4)[:, 1]

print(f'n: train={tr.sum()}  test={te.sum()}  oot={oo.sum()}   bad rate oot={100*y[oo].mean():.3f}%')

n: train=104999  test=22500  oot=22500   bad rate oot=6.684%


---
## 1. Calibration của scorecard

Một model calibrated nếu trong nhóm được chấm PD = p thì tỉ lệ vỡ nợ thực đúng bằng p. Đây là phát biểu về nhóm, không phải cá nhân: không ai kiểm được PD của một người là đúng hay sai vì người đó hoặc vỡ nợ hoặc không.

Đo bằng ba thứ. Trung bình bắt độ lệch chung. Reliability diagram chia theo decile rồi so PD dự báo với bad rate thực từng nhóm. Brier score là `mean((p−y)²)`, một con số gộp cả hai mặt.

In [2]:
rows = []
for ten, m in [('train', tr), ('test', te), ('oot', oo)]:
    mu = cal.murphy(y[m], pd_sc[m], 10)
    rows.append({'tap': ten, 'n': int(m.sum()), 'PD TB %': 100*pd_sc[m].mean(),
                 'bad rate %': 100*y[m].mean(), 'lech (diem %)': 100*(pd_sc[m].mean() - y[m].mean()),
                 'Brier': mu['brier'], 'reliability': mu['reliability'], 'resolution': mu['resolution']})
print(pd.DataFrame(rows).round(6).to_string(index=False))

  tap      n  PD TB %  bad rate %  lech (diem %)    Brier  reliability  resolution
train 104999 6.683873    6.683873       0.000000 0.050315     0.000042    0.010478
 test  22500 6.598501    6.684444      -0.085943 0.050214     0.000045    0.010406
  oot  22500 6.898120    6.684444       0.213675 0.050593     0.000054    0.010365


Trên `train` hai con số bằng nhau tới chữ số thứ sáu, đúng như phương trình chuẩn tắc của hợp lý cực đại đòi hỏi. Trên `oot` model dự báo cao hơn thực tế 0,21 điểm phần trăm. Đó là dao động lấy mẫu, không phải thiên lệch: `test` lệch theo chiều ngược lại.

### Phân rã Murphy, và một phần dư không được bỏ qua

Sách viết `Brier = Reliability − Resolution + Uncertainty`. Reliability nhỏ thì tốt, Resolution lớn thì tốt, Uncertainty là `ȳ(1−ȳ) = 0,0623` và không giảm được.

Nhưng đẳng thức đó chỉ đúng **khi p là hằng số trong từng nhóm**. Chia theo decile thì p biến thiên bên trong nhóm, nên có một phần dư. Bảng dưới cho thấy nó lớn cỡ nào.

In [3]:
print(f"{'k':>4s}{'Brier':>10s}{'reliability':>13s}{'resolution':>12s}{'du':>11s}")
for k in (5, 10, 20, 50, 100):
    mu = cal.murphy(y[oo], pd_sc[oo], k)
    print(f"{k:4d}{mu['brier']:10.5f}{mu['reliability']:13.6f}{mu['resolution']:12.5f}{mu['du']:+11.6f}")

   k     Brier  reliability  resolution         du
   5   0.05059     0.000012     0.00769  -0.004110
  10   0.05059     0.000054     0.01037  -0.001472
  20   0.05059     0.000182     0.01191  -0.000053
  50   0.05059     0.000527     0.01244  +0.000134
 100   0.05059     0.000694     0.01253  +0.000048


Hai điều rút ra, và cả hai đều là lý do không nên trích một con số Reliability trần trụi.

Phần dư ở k = 5 là −0,0041, tức gấp hơn ba trăm lần chính Reliability. Bỏ qua nó thì đọc Reliability sai bằng mấy bậc độ lớn. Nó nhỏ dần khi chia mịn hơn và gần như biến mất ở k = 50.

Nhưng Reliability lại tăng theo k, từ 0,000012 lên 0,000694. Phần lớn là vì chia mịn thì mỗi nhóm còn ít người và bad rate quan sát của nhóm nhiễu hơn. Reliability chỉ so sánh được **giữa các model trên cùng một k**.

In [4]:
# Bao nhieu phan cua Reliability la lech that, bao nhieu chi la san nhieu do chia nhom?
# Sinh nhan gia tu chinh PD cua model: khi do model calibrated tuyet doi theo dinh nghia,
# nen Reliability do duoc tren nhan gia chinh la san nhieu cua phep chia o do min k.
rng_ = np.random.default_rng(42); n_oo = int(oo.sum())
print(f"{'k':>4s}{'rel do duoc':>14s}{'san nhieu':>12s}{'ti le':>8s}")
for k in (10, 50, 100):
    rel = cal.murphy(y[oo], pd_sc[oo], k)['reliability']
    san = np.median([cal.murphy((rng_.random(n_oo) < pd_sc[oo]).astype(float), pd_sc[oo], k)['reliability']
                     for _ in range(60)])
    print(f'{k:4d}{rel:14.6f}{san:12.6f}{rel/san:8.2f}')

   k   rel do duoc   san nhieu   ti le
  10      0.000054    0.000016    3.27
  50      0.000527    0.000109    4.85
 100      0.000694    0.000222    3.12


Mức nhiễu nền tăng theo `k` gần như tỉ lệ thuận, đúng như cơ chế đã nói. Nhưng Reliability đo được **nằm trên sàn ở cả ba mức chia**, tức phần vượt sàn là độ lệch thật, không phải nhiễu, và nó khớp với độ lệch +0,21 điểm phần trăm ở bảng đầu tiên.

Nếu tỉ lệ này xấp xỉ 1 thì kết luận phải đổi: khi đó Reliability chỉ là sản phẩm của phép chia nhóm và không được dùng để nói bất cứ điều gì về calibration.

In [5]:
tab = cal.reliability(y[oo], pd_sc[oo], 10)
print((tab.assign(**{'PD du bao %': (100*tab.pd_du_bao).round(2),
                     'bad rate thuc %': (100*tab.bad_rate_thuc).round(2),
                     'lech (diem %)': (100*tab.lech).round(2)})
        [['n', 'PD du bao %', 'bad rate thuc %', 'lech (diem %)']]).to_string())

        n  PD du bao %  bad rate thuc %  lech (diem %)
dec                                                   
0    2250         0.88             0.40           0.48
1    2250         1.16             0.93           0.23
2    2250         1.46             0.93           0.53
3    2250         1.78             1.16           0.62
4    2250         2.27             1.64           0.63
5    2250         3.07             3.33          -0.26
6    2250         4.38             4.00           0.38
7    2250         6.61             6.62          -0.01
8    2250        10.86            12.44          -1.59
9    2250        36.51            35.38           1.13


Bảng này bác một nửa dự đoán A1. Tôi đoán model nén về giữa, tức đầu an toàn dự báo cao hơn thực và đầu rủi ro dự báo thấp hơn thực, đúng như đã thấy ở train và test của khối 3.

Đầu an toàn thì đúng: decile 0 dự báo 0,88% trong khi thực tế 0,40%. Đầu rủi ro thì sai chiều: decile 9 dự báo 36,51% còn thực tế 35,38%, tức cũng cao hơn. Trên `oot` model dự báo cao hơn ở 8 trên 10 decile, tức là dịch lên gần đều, không nén.

Chỗ lệch lớn nhất theo giá trị tuyệt đối nằm ở decile 8: dự báo 10,86% so với thực tế 12,44%.

---
## 2. Hai cách hiệu chỉnh, và một câu lý thuyết phải sửa

**Platt scaling** fit một logistic một chiều từ `logit(PD)` sang nhãn. Fit trên logit, không trên PD, vì trên thang log-odds thì phép hiệu chỉnh đúng là một phép dãn và dịch, còn fit trên PD thì bắt một hàm hai tham số làm thêm việc đảo ngược sigmoid của chính model.

**Isotonic regression** khớp một hàm đơn điệu bất kỳ. Linh hoạt hơn, nhưng nhiều bậc tự do hơn và ở đây chỉ có 1.504 ca dương trên `test` để fit.

`notes_credit_scoring.md` §5.3 viết: *"Cả ba đều đơn điệu nên không bao giờ đổi Gini/AUC/KS."* Đó là câu tôi sẽ kiểm, và nó là phép kiểm có đường fail rõ nhất của cả khối.

In [6]:
pl  = cal.Platt().fit(y[te], pd_sc[te])
iso = cal.Isotonic().fit(y[te], pd_sc[te])
print(f'Platt: a={pl.a:.4f}  b={pl.b:.4f}   (a=1, b=0 nghia la khong can sua gi)')

rows = []
for ten, pp in [('goc', pd_sc[oo]), ('Platt', pl.predict(pd_sc[oo])), ('isotonic', iso.predict(pd_sc[oo]))]:
    mu = cal.murphy(y[oo], pp, 10); inv = cal.rank_invariance(y[oo], pd_sc[oo], pp)
    rows.append({'phuong an': ten, 'PD TB %': 100*pp.mean(), 'Brier': mu['brier'],
                 'reliability': mu['reliability'], 'Gini': inv['gini_sau'],
                 'lech Gini': inv['lech_gini'], 'so muc PD': inv['so_muc_sau'],
                 'spearman': inv['spearman']})
print(); print(pd.DataFrame(rows).round(6).to_string(index=False))
print(f'\nbad rate oot thuc = {100*y[oo].mean():.3f}%')

Platt: a=1.0072  b=0.0327   (a=1, b=0 nghia la khong can sua gi)

phuong an  PD TB %    Brier  reliability     Gini  lech Gini  so muc PD  spearman
      goc 6.898120 0.050593     0.000054 0.712407   0.000000      10589  1.000000
    Platt 6.987520 0.050610     0.000064 0.712407   0.000000      10589  1.000000
 isotonic 6.981877 0.050167     0.000062 0.710655  -0.001752         74  0.996762

bad rate oot thuc = 6.684%


**Platt giữ Gini nguyên vẹn tới chữ số cuối và Spearman đúng bằng 1.** Đúng như lý thuyết.

**Isotonic thì không.** Nó làm mất 0,0018 Gini và Spearman rơi xuống 0,9968. Đây không phải lỗi cài đặt, và cột "số mức PD" chỉ ra ngay nguyên nhân: isotonic nén **10.589 giá trị PD phân biệt xuống còn 74**. Nó là hàm đơn điệu không nghiêm ngặt, nên nó bảo toàn thứ tự theo nghĩa yếu: hai người khác điểm trước đó có thể thành bằng điểm sau khi hiệu chỉnh. AUC tính mỗi cặp hoà là 0,5, nên Gini giảm.

Câu trong ghi chú nền tảng vì thế phải sửa: **Platt không bao giờ đổi thứ hạng; isotonic thì có, qua đường tạo hoà.** Cái giá ở đây là 0,0018 Gini, nhỏ nhưng có thật và đo được.

Còn về calibration, kết quả ngược với dự đoán A5:

- Platt làm Brier tệ đi một chút (0,050593 → 0,050610).
- Isotonic làm Brier tốt lên (0,050593 → 0,050167), tức mua calibration bằng cách bán một ít khả năng xếp hạng.

Lý do Platt làm tệ đi đáng ghi hơn bản thân con số. Trên `test` model dự báo thấp hơn bad rate (6,599% so với 6,684%), nên Platt học được phép "kéo lên" (b = +0,033). Trên `oot` model đã dự báo cao hơn (6,898%), nên kéo lên nữa là kéo sai chiều.

**Một lớp hiệu chỉnh fit trên kỳ này áp lên kỳ khác chỉ giúp nếu độ lệch giữ nguyên dấu.** Ở đây nó đổi dấu, và cả hai lần đổi đều chỉ là dao động lấy mẫu quanh 0. Hiệu chỉnh một model vốn đã calibrated là thêm nhiễu, không thêm thông tin.

---
## 3. Bất đối xứng: calibration sửa được sau, xếp hạng thì không

`notes_credit_scoring.md` §5.3 nói lựa chọn giữa model A xếp hạng giỏi nhưng calibrate tệ và model B ngược lại thật ra là lựa chọn giữa **A đã hiệu chỉnh** và B. Ở đây có sẵn hai model để kiểm.

In [7]:
rows = []
for ten, p_ in [('scorecard đơn điệu', pd_sc), ('XGBoost M4', pd_xgb)]:
    mu = cal.murphy(y[oo], p_[oo], 10)
    from sklearn.metrics import roc_auc_score
    rows.append({'model': ten, 'Gini oot': 2*roc_auc_score(y[oo], p_[oo])-1,
                 'Brier oot': mu['brier'], 'reliability': mu['reliability'],
                 'resolution': mu['resolution'], 'PD TB %': 100*p_[oo].mean()})
print(pd.DataFrame(rows).round(6).to_string(index=False))

             model  Gini oot  Brier oot  reliability  resolution  PD TB %
scorecard đơn điệu  0.712407   0.050593     0.000054    0.010365 6.898120
        XGBoost M4  0.729663   0.049028     0.000019    0.011375 6.876384


XGBoost trội scorecard ở cả hai mặt cùng lúc: Gini cao hơn 0,017 và Brier thấp hơn 0,0016. Dự đoán A6 trúng.

Nhìn vào phân rã thì thấy vì sao. Reliability của hai model đều rất nhỏ (0,000019 so với 0,000054), tức cả hai đều calibrate tốt như nhau. Toàn bộ chênh lệch Brier đến từ Resolution: 0,01138 so với 0,01037. Mà Resolution chính là khả năng phân biệt, tức đúng thứ Gini đo.

Nói cách khác: ở bộ này Brier không mang thông tin nào mà Gini chưa có. Nó chỉ hữu ích khi Reliability khác nhau đáng kể giữa hai model, và điều đó chỉ xảy ra khi có một model bị phá calibration, ví dụ bằng `scale_pos_weight` như đã thấy ở khối 4.

---
## 4. PSI: quần thể có còn giống lúc build không

```
PSI = SUM_i (a_i − e_i) · ln(a_i / e_i)
```

`e_i` là tỉ trọng bin i ở mẫu tham chiếu (train lúc build), `a_i` ở mẫu mới. Từng số hạng không âm nên PSI ≥ 0. Ngưỡng quy ước: dưới 0,1 ổn định, 0,1 đến 0,25 theo dõi, trên 0,25 điều tra. Đây là ngưỡng kinh nghiệm, không có nền tảng lý thuyết.

Điều quan trọng nhất về cài đặt: **ranh giới bin phải đóng băng**. Tính một lần trên train rồi dùng y nguyên mãi mãi. Mục 4.3 đo hậu quả của việc làm sai chỗ này.

`oot` ở đây là lát cắt ngẫu nhiên từ cùng quần thể nên nó không đo được drift nào. Cái nó đo được là ba thứ khác: pipeline áp bin có đúng không, mức nền là bao nhiêu để ngưỡng 0,1 có nghĩa, và công thức có cài đúng không.

In [8]:
cuts = P.freeze_cuts(pd_sc[tr], 10)          # dong bang tren train, khong bao gio tinh lai
for ten, m in [('train -> test', te), ('train -> oot', oo)]:
    r = P.psi(pd_sc[tr], pd_sc[m], cuts)
    print(f'{ten:16s} PSI = {r["psi"]:.6f}   ({r["n_bin"]} bin, {r["bin_rong_moi"]} bin rong)')
print(f'{"xap xi ly thuyet":16s} PSI ~ {9*(1/tr.sum()+1/oo.sum()):.6f}   = (k-1)(1/n1+1/n2)')

train -> test    PSI = 0.000372   (10 bin, 0 bin rong)
train -> oot     PSI = 0.001155   (10 bin, 0 bin rong)
xap xi ly thuyet PSI ~ 0.000486   = (k-1)(1/n1+1/n2)


Cả hai đều xa dưới 0,1. Nhưng hai con số chênh nhau ba lần (0,00037 và 0,00116) trong khi cả hai đều là mẫu ngẫu nhiên từ cùng quần thể, nên câu hỏi đúng không phải "PSI bằng bao nhiêu" mà **"PSI dao động cỡ nào khi không có drift"**. Xấp xỉ lý thuyết cho một con số duy nhất; đo trực tiếp thì cho cả phân phối.

In [9]:
rng = np.random.default_rng(0); st = pd_sc[tr]; v = []
for _ in range(200):
    idx = rng.choice(len(st), int(oo.sum()), replace=False)
    m = np.zeros(len(st), bool); m[idx] = True
    v.append(P.psi(st[~m], st[m], cuts)['psi'])
v = np.array(v)
r_oot = P.psi(pd_sc[tr], pd_sc[oo], cuts)['psi']
n1, n2 = len(st) - int(oo.sum()), int(oo.sum())
print(f'200 lan cat ngau nhien chinh train, cung co mau voi oot ({n1} so voi {n2}):')
print(f'   trung binh = {v.mean():.6f}   trung vi = {np.median(v):.6f}   p95 = {np.quantile(v, .95):.6f}   max = {v.max():.6f}')
print(f'   ly thuyet cho dung cap co mau nay = {9*(1/n1+1/n2):.6f}')
print(f'   PSI cua oot ({r_oot:.6f}) nam o phan vi {100*(v < r_oot).mean():.0f}%')

200 lan cat ngau nhien chinh train, cung co mau voi oot (82499 so voi 22500):
   trung binh = 0.000510   trung vi = 0.000465   p95 = 0.000956   max = 0.001579
   ly thuyet cho dung cap co mau nay = 0.000509
   PSI cua oot (0.001155) nam o phan vi 98%


Trung bình mô phỏng là 0,000510 so với xấp xỉ lý thuyết 0,000509 cho đúng cặp cỡ mẫu đó, lệch 0,3%. **Công thức cài đúng**, và đây mới là phép so đúng: xấp xỉ `(k−1)(1/n₁+1/n₂)` xấp xỉ kỳ vọng của PSI, mà PSI xấp xỉ một chi-bình phương 9 bậc tự do chia cho cỡ mẫu nên phân phối của nó lệch phải. Vì vậy trung vị 0,000465 phải nằm THẤP HƠN trung bình đúng bằng độ lệch của chi-bình phương: tỉ lệ lý thuyết là 0,927, đo được 0,911.

Nhưng phân phối rộng: từ dưới 0,0002 lên tới 0,0016 chỉ vì lấy mẫu. PSI của `oot` nằm ở phân vị 98%, tức hơi cao nhưng vẫn nằm trong dải mà chỉ nhiễu cũng tạo ra được. Với một lần đo thì không kết luận được gì, và đó chính là điều đáng ghi: **một con số PSI đơn lẻ không đọc được nếu không biết mức nền của chính hệ thống đó.** Ngưỡng 0,1 an toàn ở đây vì nó cách mức nền hai bậc độ lớn, không phải vì nó có cơ sở lý thuyết.

In [10]:
con = sqlite3.connect(scorecard._ro_uri(config.DB_PATH), uri=True)
long = pd.read_sql('SELECT id, variable, bin FROM row_bins_mono', con); con.close()
sp = bins_m.split
long = long.assign(split=long.id.map(sp))
tab = P.csi_table(long[long.split == 'train'], long[long.split == 'oot'])
print(tab.round(6).to_string(index=False))

         variable      csi
       late_60_89 0.000536
              age 0.000373
       late_30_59 0.000372
   monthly_income 0.000215
 debt_ratio_valid 0.000207
   revolving_util 0.000183
          late_90 0.000131
       dependents 0.000047
real_estate_loans 0.000001


CSI của cả chín biến đều dưới 0,0006, thấp hơn cả PSI của điểm số. Dự đoán B2 trúng và còn chặt hơn ngưỡng tôi đặt.

Điều này xác nhận thứ nó sinh ra để xác nhận: ranh giới bin của khối 2 và bảng tra WOE được áp sang `oot` **đúng như đã áp cho train**. Nếu một biến nào vượt 0,02 thì đó là bug ở bước JOIN, không phải drift, vì `oot` không thể drift so với train khi nó là lát cắt ngẫu nhiên.

---
## 4.2 OOT dịch nhân tạo

`oot` không đo được drift nên PSI ở trên không chứng minh được là nó bắt được drift. Cách kiểm: dựng một quần thể đã dịch có chủ ý, lấy mẫu lại `oot` thiên về nhóm `revolving_util` cao, mô phỏng một chiến dịch mang về khách rủi ro hơn.

Trọng số `w = exp(α · phân vị(util))`, lấy mẫu không hoàn lại 12.000 trên 22.500 dòng. Điều quan trọng nhất: trọng số **chỉ phụ thuộc X, tuyệt đối không nhìn y**. Nhờ vậy phép lấy mẫu đổi P(X) mà giữ nguyên P(y|X), và điều đó cho phép kiểm một mệnh đề cụ thể ở mục sau.

In [11]:
d_oot = pd.DataFrame({'util': ap.revolving_util.values[oo], 'score': pd_sc[oo],
                      'xgb': pd_xgb[oo], 'y': y[oo]})
from sklearn.metrics import roc_auc_score
rows = []
for a in (0, 1, 2, 3, 4, 6):
    s_ = P.shift_sample(d_oot, 'util', a, 12000, seed=config.SEED)
    r = P.psi(pd_sc[tr], s_.score.values, cuts)
    rows.append({'alpha': a, 'PSI': r['psi'], 'bad rate %': 100*s_.y.mean(),
                 'PD TB %': 100*s_.score.mean(), 'lech (diem %)': 100*(s_.score.mean()-s_.y.mean()),
                 'Gini': 2*roc_auc_score(s_.y, s_.score)-1,
                 'util trung vi': s_.util.median()})
print(pd.DataFrame(rows).round(4).to_string(index=False))

 alpha    PSI  bad rate %  PD TB %  lech (diem %)   Gini  util trung vi
     0 0.0023      6.6583   6.8985         0.2401 0.7134         0.1585
     1 0.0298      7.7000   8.1675         0.4675 0.7053         0.2525
     2 0.0921      8.9833   9.1679         0.1846 0.6941         0.3444
     3 0.1793      9.6667   9.9901         0.3234 0.6801         0.4293
     4 0.2660      9.9833  10.5132         0.5298 0.6848         0.4789
     6 0.4145     10.7000  11.0242         0.3242 0.6595         0.5213


Đây là bảng đáng giá nhất khối này, và nó xác nhận dự đoán B4 ở cả bốn vế.

**PSI báo động.** Ở α = 6 nó lên 0,41, tức vượt xa ngưỡng điều tra 0,25 và gấp hơn ba trăm lần mức nền 0,0012. Chuông báo cháy hoạt động.

**Quần thể đúng là rủi ro hơn thật.** Bad rate thực đi từ 6,66% lên 10,70%, tức gấp 1,6 lần.

**Nhưng model không hề nói dối.** PD dự báo trung bình đi từ 6,90% lên 11,02%, bám sát bad rate thực ở mọi mức dịch, độ lệch lớn nhất là 0,53 điểm phần trăm ở α = 4. Lý do là phép lấy mẫu chỉ đụng P(X); P(y|X) giữ nguyên, mà calibration là phát biểu về P(y|X).

**Cái mất là xếp hạng.** Gini rơi từ 0,713 xuống 0,660. Quần thể mới tập trung vào một vùng hẹp của biến mạnh nhất nên còn ít thứ để phân biệt.

Kết luận vận hành, và nó là lý do PSI tồn tại chứ cũng là giới hạn của PSI: **PSI vượt ngưỡng có nghĩa là "quần thể đã đổi, hãy đi xem", không có nghĩa là "model đã sai".** Ở đây model vẫn nói đúng PD của từng người; thứ xuống cấp là khả năng xếp hạng, mà PSI không đo được điều đó. Trong công thức PSI không có `y` ở bất kỳ đâu.

---
## 4.3 Cái bẫy: bin không đóng băng

Lỗi cài đặt phổ biến nhất của PSI là mỗi kỳ giám sát lại chia decile mới trên dữ liệu mới. Khi đó tỉ trọng quan sát luôn bằng 0,10 ở mọi bin và PSI luôn bằng 0 dù quần thể dịch tới đâu: ta đang đo phân phối của dữ liệu mới so với chính nó.

Cái bẫy này im lặng tuyệt đối, dashboard xanh mượt trong khi model đang chết. Dưới đây là nó, bằng số, trên đúng quần thể vừa chứng minh là đã dịch mạnh.

In [12]:
s6 = P.shift_sample(d_oot, 'util', 6, 12000, seed=config.SEED)
nb_ = len(cuts) + 1

# (1) DUNG: ranh gioi dong bang tu train, ap y nguyen cho ca hai ben
psi_dung = P.psi(pd_sc[tr], s6.score.values, cuts)['psi']

# (2) CAI BAY: moi ky tu chia decile cua chinh minh, nen ca e lan a deu bang 0,10
e_bay = P.shares(P.bin_by_cuts(pd_sc[tr], P.freeze_cuts(pd_sc[tr], 10)), nb_)
a_bay = P.shares(P.bin_by_cuts(s6.score.values, P.freeze_cuts(s6.score.values, 10)), nb_)
psi_bay = P.psi_from_shares(e_bay, a_bay)

# (3) Bien the: lay ranh gioi cua ky MOI roi ap cho ca hai. Van bat duoc, vi tham
#     chieu khong con deu nua. Cai bay nam o cho de moi ben tu chia, khong phai o
#     cho tinh lai ranh gioi.
cuts_moi = P.freeze_cuts(s6.score.values, 10)
psi_bt = P.psi(pd_sc[tr], s6.score.values, cuts_moi)['psi']

print(f'(1) ranh gioi dong bang tu train      : PSI = {psi_dung:.6f}')
print(f'(2) moi ky tu chia decile cua minh    : PSI = {psi_bay:.6f}   <- CAI BAY')
print(f'(3) lay ranh gioi ky moi, ap cho ca hai: PSI = {psi_bt:.6f}')
print(f'\nti trong tham chieu o (2): {np.round(e_bay, 4)}')
print(f'ti trong ky moi   o (2): {np.round(a_bay, 4)}')
print(f'\nmuc nen (oot ngau nhien, ranh gioi dong bang) = {r_oot:.6f}')

(1) ranh gioi dong bang tu train      : PSI = 0.414452
(2) moi ky tu chia decile cua minh    : PSI = 0.000004   <- CAI BAY
(3) lay ranh gioi ky moi, ap cho ca hai: PSI = 0.405777

ti trong tham chieu o (2): [0.1    0.1001 0.0998 0.1    0.1    0.1    0.1001 0.0999 0.1    0.1   ]
ti trong ky moi   o (2): [0.1002 0.0998 0.1001 0.1003 0.0996 0.1    0.1    0.1    0.1    0.1   ]

muc nen (oot ngau nhien, ranh gioi dong bang) = 0.001155


Cùng một quần thể đã chứng minh là dịch mạnh, cùng một công thức, ba cách chia bin cho ba kết quả khác hẳn nhau.

Cách đúng cho 0,41. Cái bẫy cho một con số **nhỏ hơn mức nền của một quần thể không hề dịch chuyển**: khi mỗi kỳ tự chia decile của mình thì cả tỉ trọng tham chiếu lẫn tỉ trọng kỳ mới đều bằng 0,10 ở mọi bin theo đúng cấu tạo, nên PSI không còn đo được gì. Chỉ số giám sát vẫn nằm yên trong ngưỡng an toàn trong khi quần thể đã dịch chuyển hoàn toàn.

Cách thứ ba đáng chú ý vì nó vẫn bắt được (0,41). Nghĩa là cái bẫy không nằm ở chỗ tính lại ranh giới, mà ở chỗ **để mỗi bên tự chia theo phân phối của mình**. Chừng nào cả hai bên còn dùng chung một bộ ranh giới thì PSI vẫn đo được thứ nó cần đo; hỏng là khi mẫu tham chiếu bị chia lại theo chính nó.

Đây là chỗ dự đoán B5 của tôi trúng kết quả nhưng lần đầu tôi mô phỏng sai cái bẫy: tôi lấy ranh giới của kỳ mới rồi áp cho cả hai bên, tức làm ra cách thứ ba, và ra 0,41, không phải 0. Con số không khớp dự đoán là thứ chỉ ra tôi cài sai, nên tôi giữ cả ba dòng thay vì xoá dòng sai đi.

Ranh giới bin là tham số của hệ thống giám sát, và tham số thì đóng băng cùng model. Đó là lý do `psi()` trong `src/psi.py` bắt buộc nhận `cuts` từ ngoài, không tự tính.

---
## 5. Thống nhất khoảng tin cậy của Gini

Khối 3 dùng ±0,028 cho `oot`, khối 4 dùng ±0,025 cho `test`. Hai tập có đúng cùng số ca dương nên hai con số không thể khác nhau vì dữ liệu. Mục này đo lại và chốt một con số.

In [13]:
from sklearn.metrics import roc_auc_score
def hanley(y_, p_):
    A = roc_auc_score(y_, p_); n1 = int(np.sum(y_ == 1)); n2 = int(np.sum(y_ == 0))
    Q1, Q2 = A/(2-A), 2*A*A/(1+A)
    se = np.sqrt((A*(1-A) + (n1-1)*(Q1-A*A) + (n2-1)*(Q2-A*A)) / (n1*n2))
    return 2*A-1, 1.96*2*se, n1, n2

for ten, m in [('test', te), ('oot', oo)]:
    g, hw, n1, n2 = hanley(y[m], pd_sc[m])
    rg = np.random.default_rng(config.SEED); bs = []
    yy, pp_ = y[m], pd_sc[m]
    for _ in range(500):
        i = rg.integers(0, len(yy), len(yy)); bs.append(2*roc_auc_score(yy[i], pp_[i])-1)
    print(f'{ten:5s} Gini={g:.4f}  {n1} ca duong  Hanley-McNeil +-{hw:.4f}  bootstrap 500 lan +-{1.96*np.std(bs, ddof=1):.4f}')

test  Gini=0.7006  1504 ca duong  Hanley-McNeil +-0.0247  bootstrap 500 lan +-0.0215
oot   Gini=0.7124  1504 ca duong  Hanley-McNeil +-0.0243  bootstrap 500 lan +-0.0198


Hai tập cho gần đúng cùng một con số, như phải thế. Chốt ±0,025 cho cả dự án, tính bằng Hanley–McNeil, và ghi kèm bootstrap ±0,021 làm mốc thứ hai.

Chênh lệch với ±0,028 của khối 3 không phải hai cách tính khác nhau: bảng ở `notes_credit_scoring.md` §4.9 viết lúc chưa có model nên đặt **AUC giả định 0,78**, mà Hanley–McNeil cho SE nhỏ dần khi AUC lớn dần. Cùng công thức, cùng 1.504 ca dương: AUC 0,78 cho ±0,0281, AUC 0,856 thật cho ±0,0243.

Mọi chênh lệch Gini dưới 0,025 trên một tập giữ riêng ở dự án này là **chưa kết luận được**, và đó là lý do so sánh scorecard với XGBoost phải làm bằng CV ghép cặp trong train, không bằng cách nhìn hai con số test.

---
## Xong bước này

| | |
|---|---|
| Calibration scorecard trên `oot` | PD TB 6,90% so với bad rate 6,68%, lệch 0,21 điểm phần trăm |
| Brier | scorecard 0,0506, XGBoost 0,0491 |
| Platt | giữ Gini chính xác, Brier tệ đi 0,00002 |
| Isotonic | Brier tốt lên 0,00043 nhưng mất 0,0018 Gini |
| Bất đối xứng | XGBoost trội scorecard ở cả Gini lẫn Brier, chênh lệch nằm hết ở Resolution |
| PSI mức nền | 0,0012 trên `oot`, phân phối mô phỏng 0,0002 đến 0,0016 |
| CSI chín biến | đều dưới 0,0006 |
| OOT dịch (α = 6) | PSI 0,41, bad rate 10,7%, PD dự báo 11,0%, Gini rơi 0,71 → 0,66 |
| Bin không đóng băng | cùng quần thể đó cho PSI 0,000004, tức im lặng hoàn toàn |
| KTC 95% của Gini | chốt ±0,025 (Hanley–McNeil), bootstrap ±0,021 |

**Mười bốn dự đoán ở `notes/du_doan_khoi5.md`: mười trúng, ba trượt, một trúng một nửa.** Chi tiết và cách chấm ở `results/calibration_psi.md`.

Ba cái trượt đáng nhìn hơn mười cái trúng. A4 và A5 đều nói sai về hiệu chỉnh: tôi tưởng isotonic giữ nguyên thứ hạng (nó không), và tưởng Platt sẽ cải thiện Brier (nó làm tệ đi). B6 thì trượt vì đoán Gini bi quan, **lần thứ tư liên tiếp** trong dự án, dù lần này tôi đã ghi rõ trong file dự đoán rằng mình biết thói quen đó và đã cố đặt khoảng cao hơn.

Hai thứ khối này lật lại:

- Câu *"cả ba cách hiệu chỉnh đều đơn điệu nên không bao giờ đổi Gini/AUC/KS"* ở `notes_credit_scoring.md` §5.3 sai với isotonic. Nó đơn điệu không nghiêm ngặt, nén 10.589 mức PD xuống 74, tạo hoà hàng loạt và làm mất 0,0018 Gini.
- Giả thuyết số 2 ghi từ khối 0 (*"XGBoost vượt scorecard 0,02 đến 0,06 Gini"*) sai: đo được 0,0143 trên cùng 9 biến ở khối 4.

Việc còn lại cho khối 6: chia bin lại cho `open_credit_lines` rồi cân nhắc đưa nó trở lại scorecard, vì khối 4 đã đo được nó mang hiệu ứng chính đáng 0,002 đến 0,003 Gini mà dạng hàm của scorecard không dùng được.